# Real-ESRGAN — Video Enhancement Pipeline

## What this notebook demonstrates

This notebook enhances video quality using **Real-ESRGAN** (Enhanced Super-Resolution Generative Adversarial Network), a neural network that restores and upscales images by learning to reverse common degradations: compression artefacts, noise, blur, and low resolution.

### How Real-ESRGAN works

Real-ESRGAN is a **GAN** (Generative Adversarial Network). GANs consist of two competing networks trained together:
- The **generator** learns to produce high-quality output from degraded input.
- The **discriminator** learns to tell real high-quality images from generated ones.

This adversarial training forces the generator to produce outputs that are perceptually indistinguishable from real sharp images. The "Real" in Real-ESRGAN refers to the use of a **realistic degradation model** during training — mixing blur, noise, JPEG compression, and resizing in combinations that match real-world footage — so the model generalises far beyond clean synthetic test data.

The model used here (`RealESRGAN_x4plus`) was pre-trained on natural images and outputs at **4× the input resolution** (controllable via `--outscale`).

> **What "restoration" actually means:** Real-ESRGAN does not recover information that was never there. It uses patterns learned during training to *hallucinate* plausible high-frequency detail — sharpening edges, removing blocking from video compression, recovering texture. The result is a plausible reconstruction, not a ground-truth recovery.

### Two-pass workflow

This notebook supports two modes of use:

**Single pass (restoration only):** run the notebook top to bottom.
```
Extract frames → Restore with ESRGAN → Reassemble → Reattach audio
```

**Two-pass (restoration + interpolation):**
```
Pass 1 (this notebook):        Extract → Restore → Save to Drive
Interpolation notebook:        Load from Drive → RIFE interpolation → Save to Drive
Pass 2 (this notebook):        Load from Drive → (Optional) Superscale → Reassemble → Reattach audio
```

**Steps in this notebook:**
1. Install dependencies and mount Google Drive
2. Upload video; extract frames and analyse frame rate (CFR/VFR check)
3. Set up Real-ESRGAN
4. Restore frames (denoising, deblurring, upscaling)
5. Prepare frames for interpolation (optional: downscale + save to Drive)
6. *[Separate notebook]* Frame interpolation with RIFE
7. Load processed frames back from Drive (optional, after interpolation)
8. Reassemble video and reattach audio

## ** Install dependencies

| Package | Role |
|---|---|
| `ffmpeg-python` | Python bindings for FFmpeg. Used to extract frames from the video and reassemble the output. |
| `ffmpeg` (system) | The actual `ffmpeg` and `ffprobe` binaries — `ffmpeg-python` is only a wrapper around these. |

Real-ESRGAN and its dependencies (`basicsr`, `facexlib`, `gfpgan`) are installed directly from GitHub in Step 3, not from pip at this stage.

> **Runtime note:** Installing Real-ESRGAN changes the working directory to `/content/Real-ESRGAN`. All subsequent cells that reference frames or output use **absolute paths** (e.g. `/content/frames/`) to avoid confusion.

In [ ]:
# 📦 Install dependencies
%pip install ffmpeg-python
!apt-get install -y ffmpeg

## ** Create working folders and mount Google Drive

Colab runtimes are **ephemeral** — all local files are lost when the session ends or disconnects. Google Drive provides persistent storage so that:
- The input video survives a runtime restart.
- Restored frames can be handed off to a separate interpolation notebook.
- The final output is preserved after the session ends.

The Rekrea folder structure created in Drive:
```
MyDrive/Rekrea/
  input_video.mp4                        ← source video (uploaded once, reused across sessions)
  VideoRestoration/
    output/                              ← restored frames archive (restored_frames_fps_N.tar.gz)
    downscaled/                          ← downscaled frames archive (for interpolation handoff)
  VideoInterpolation/
    output/                              ← interpolated frames archive (from RIFE notebook)
```

In [ ]:
# 📁 Create runtime folders
import os
for folder in ["frames", "restored_frames"]:
    os.makedirs(folder, exist_ok=True)

# 📂 Mount Google Drive to setup & access environment
from google.colab import drive
drive.mount('/content/drive')

## Step 1 — Upload your video

The upload cell implements a **smart caching pattern** to avoid re-uploading the same video across sessions:

```
Does input_video.mp4 exist in Drive (Rekrea folder)?
  ├─ No  → Prompt upload → save to Drive for future sessions
  └─ Yes → Skip upload (reuse the persisted copy)

Does input_video.mp4 exist locally in the Colab runtime?
  ├─ No  → Copy from Drive to local runtime
  └─ Yes → Already available, nothing to do
```

This matters because Colab runtimes are ephemeral — uploading a large video file every session is slow and unnecessary if the file already lives in Drive.

**Tips for best results:**
- Use an MP4 file with H.264 video — the most compatible input format for FFmpeg.
- Shorter clips (5–30 seconds) are recommended to keep processing time manageable.
- The video filename is fixed to `input_video.mp4` inside the notebook. If you want to process a different clip, delete or rename the existing one in the Rekrea Drive folder before uploading.
- If your video was recorded on a phone it may use a **Variable Frame Rate (VFR)** — this will be detected and flagged in Step 2.2.

In [ ]:
# ⬆️ Step 1: Upload your video
from google.colab import files
from pathlib import Path

INPUT_VIDEO = "input_video.mp4"
ENV_DIR = Path("/content/drive/MyDrive/Rekrea")
DRV_INPUT_PATH = ENV_DIR  / INPUT_VIDEO
LOCAL_INPUT_PATH = Path("/content") / INPUT_VIDEO

# Create Rekrea folder in Drive if it doesn't exist
ENV_DIR .mkdir(parents=True, exist_ok=True)

# Check if input video exits in Rekrea environment and upload 
# Otherwise request to upload and save in Rekrea enviroment
if not DRV_INPUT_PATH.exists():
    uploaded_file = files.upload()
    original_name = list(uploaded_file.keys())[0]
    os.rename(original_name, INPUT_VIDEO)
    !cp "{INPUT_VIDEO}" "{DRV_INPUT_PATH}"
    print("Uploaded video stored in Rekrea environment in Drive.")
    
print("Persistent input stored in Rekrea environment in Drive.")

if not LOCAL_INPUT_PATH.exists():
    !cp "{DRV_INPUT_PATH}" "{LOCAL_INPUT_PATH}"
    print("Uploading from Rekrea environment in Drive to Colab runtime")

print("Local working copy ready.")

## Step 2.1 — Extract frames and analyse frame rate

Steps 2.1 and 2.2 are intentionally separated for clarity but must be run together before proceeding.

**Step 2.1** decodes the video into individual PNG frames using FFmpeg.

**Step 2.2** performs a critical quality check: it determines whether the video uses a **Constant Frame Rate (CFR)** or a **Variable Frame Rate (VFR)**.

- **CFR:** every frame has the same duration — safe to reassemble with a fixed `framerate=` parameter.
- **VFR:** frame durations vary — common in screen recordings and phone videos. Reassembling a VFR video with a fixed frame rate produces a video with incorrect timing (frames appear too fast or too slow).

The check cross-references three independent measurements:
1. FPS and frame count from OpenCV container metadata.
2. Container duration from FFprobe.
3. Actual number of PNG files extracted to disk.

If these disagree by more than 1%, VFR is suspected and `cfr` is set to `None`. The reassembly step raises an explicit error rather than silently producing a broken video.

In [ ]:
# 🎞️ Step 2.1: Extract frames from the video
import ffmpeg

INPUT_VIDEO = "input_video.mp4" 

# Extract frames from uploaded video
(
    ffmpeg
    .input(INPUT_VIDEO)
    .output('frames/frame_%05d.png', pix_fmt='rgb24', qscale=1)
    .global_args("-vsync", "0")
    .run()
)

In [ ]:
# 〰️ Step 2.2: Determine frame rate in fps
import cv2, subprocess, json, math, os

INPUT_VIDEO = "input_video.mp4" 

# Functions to check against VFR and determine CFR
def ffprobe_duration(path: str) -> float | None:
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "json", path]
    out = subprocess.check_output(cmd).decode("utf-8")
    dur = json.loads(out).get("format", {}).get("duration", None)
    return float(dur) if dur is not None else None

def video_intake_check(video_path: str, frames_folder: str, rel_tol: float = 0.01):
    # OpenCV metadata
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames_metadata = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    fps = float(fps) if fps and fps > 0 else None
    n_frames_metadata = int(n_frames_metadata) if n_frames_metadata and n_frames_metadata > 0 else None

    # count from extracted frames
    n_frames_data = sum(1 for f in os.listdir(frames_folder) if f.endswith(".png"))

    # ffprobe container duration
    dur = ffprobe_duration(video_path)

    # Derived duration from frames/fps
    dur_calc_metadata = (n_frames_metadata / fps) if (fps and n_frames_metadata) else None
    dur_calc_data = (n_frames_data / fps) if (fps and n_frames_data) else None

    # Test if VFR
    vfr_suspected = None
    if dur is not None and dur_calc_data is not None and dur > 0:
        vfr_suspected = (abs(dur_calc_data - dur) / dur) > rel_tol

    # Determine CFR value
    cfr = None
    if vfr_suspected is False and fps is not None:
        cfr = fps

    return {
        "fps_metadata": fps,
        "n_frames_metadata": n_frames_metadata,
        "n_frames_extracted": n_frames_data,
        "duration_ffprobe": dur,
        "duration_calc_frames_over_fps": dur_calc_data,
        "vfr_suspected": vfr_suspected,
        "cfr": cfr
    }

# Perform the frame rate analysis
report = video_intake_check(INPUT_VIDEO, "frames")
print(report)

# Extract the frame rate in fps
# If "None" assume VFR and normalize (TODO) 
cfr = report["cfr"]

## Step 3 — Set up Real-ESRGAN

Real-ESRGAN is installed from its GitHub repository rather than PyPI because it requires a `setup.py develop` install to resolve its internal package structure (`basicsr`).

**What gets installed:**
- `basicsr` — the underlying super-resolution framework by the same authors.
- `facexlib` / `gfpgan` — face restoration libraries (used when `--face_enhance` is passed).
- Pre-trained weights (`RealESRGAN_x4plus.pth`, ~67 MB) downloaded from the GitHub release.

**The torchvision patch** on the last line is required because recent versions of `torchvision` moved `rgb_to_grayscale` to a new module path. The `sed` command patches the import in `basicsr/data/degradations.py` so it works with Colab's current environment without modifying the Real-ESRGAN source directly.

> **Alternative pre-trained models** available in the Real-ESRGAN GitHub releases:
>
> | Model | Best for |
> |---|---|
> | `RealESRGAN_x4plus` | General photos and video footage |
> | `RealESRGAN_x4plus_anime_6B` | Anime and illustration content (lighter, 6 basic blocks) |
> | `RealESRNet_x4plus` | Smoother, less aggressive output — fewer GAN artefacts |
> | `RealESRGAN_x2plus` | When 2× upscaling is sufficient |

In [ ]:
# 🧠 Step 3: Set Up Real-ESRGAN and patch torchvision issue
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
%pip install -r requirements.txt
!python setup.py develop
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights

# Patch torchvision change
!sed -i "s/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.v2 import functional as F\nrgb_to_grayscale = F.rgb_to_grayscale/" /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

## Step 4 — Restore frames with Real-ESRGAN

`inference_realesrgan.py` runs the generator network on every frame. Key parameters:

| Parameter | Value | Meaning |
|---|---|---|
| `-n RealESRGAN_x4plus` | model name | Which pre-trained weights to load |
| `-i /content/frames/` | input dir | Where raw frames are read from |
| `-o /content/restored_frames/` | output dir | Where restored frames are written |
| `--outscale 2` | 2× | Final output scale. The model works at 4× internally then downscales. Use 2× unless you have >16 GB VRAM. |
| `--face_enhance` | flag | Activates GFPGAN to additionally restore facial features. Omit for content without close-up faces — it adds processing time. |

After processing, VRAM is explicitly released with `torch.cuda.empty_cache()`. This is important if interpolation follows in the same session — it frees memory for the next model.

In [ ]:
# 🚀 Step 4: Restore frames
!python inference_realesrgan.py -n RealESRGAN_x4plus -i /content/frames/ -o /content/restored_frames/ --outscale 2 --face_enhance

# Release GPU Memory
import torch
torch.cuda.empty_cache()

## Step 5.1 — Rename restored frames

Real-ESRGAN appends `_out` to every output filename:
```
frame_00001.png  →  frame_00001_out.png
```

Later steps — FFmpeg reassembly, the RIFE interpolation notebook, and superscaling — all expect the clean pattern `frame_XXXXX.png`. This cell strips the `_out` suffix in-place to restore the expected naming convention before any further processing.

In [ ]:
# 🪄 Step 5.1: Rename Real-ESGRAN restored frames 
from pathlib import Path

LOCAL_RES_DIR = Path("/content/restored_frames")

# ESRGAN often outputs: frame_00001_out.png
# We'll convert to: reconstructed_frames/frame_00001.png  (clean pattern for later steps)
for file in LOCAL_RES_DIR.glob("frame_*_out.png"):
    new_name = file.name.replace("_out", "")
    new_path = file.with_name(new_name)
    file.rename(new_path)

print("All frames renamed.")

## Step 5.2a — Prepare frames for interpolation (optional)

The following two steps are only needed if you plan to run **frame interpolation** in a separate notebook.

**Decision tree:**
```
Restoration complete
  └─ Want smoother motion (frame interpolation)?
      ├─ No  → Skip to Step 8 (Reassemble Final Video)
      └─ Yes → Run Step 5.2a (downscale) + Step 5.2b (save to Drive)
               → Open the VFIformer interpolation notebook
               → Return here at Step 7 to load the interpolated frames
```

**Why downscale before interpolation (Step 5.2a)?**
Real-ESRGAN outputs at 2× or 4× the original resolution. VFIformer processes frame pairs and its VRAM cost scales with resolution. Downscaling the restored frames to HD before interpolation:
- Keeps VRAM usage manageable on Colab GPUs.
- Speeds up interpolation significantly.
- Allows a second ESRGAN pass (superscaling) afterwards to recover resolution.

**Why 704×1280 and not 720×1280 (standard HD portrait)?**

VFIformer's transformer attention layers require both the width and height of each frame to be **divisible by 32**. This is a hard constraint: if either dimension is not a multiple of 32, the model pads the input internally, which can produce artefacts at the frame edges.

| Dimension | Value | ÷ 32 | Valid? |
|---|---|---|---|
| 720 (standard HD) | 720px | 22.5 | ✗ Not divisible |
| **704 (used here)** | **704px** | **22** | **✓ Divisible** |
| 1280 | 1280px | 40 | ✓ Divisible |

`704×1280` is the closest HD portrait resolution where **both** dimensions are divisible by 32. The 16-pixel difference (720 → 704) is imperceptible in the final video and eliminates a source of subtle edge artefacts inside VFIformer.

> This constraint applies equally to RIFE and other transformer-based interpolation models — it is a property of how self-attention tiles the spatial dimensions, not specific to VFIformer.

In [ ]:
# 🪄 Step 5.2a: Downscale restored frames BEFORE compressing (batch, high quality, optional) 
from pathlib import Path

# Source and destination directories
LOCAL_RES_DIR = Path("/content/restored_frames")
LOCAL_DS_DIR = Path("/content/restored_frames_ds")
LOCAL_DS_DIR.mkdir(parents=True, exist_ok=True)

# Target 704x1280: both dimensions divisible by 32, required by VFIformer's attention layers.
# 720x1280 (standard HD portrait) is NOT valid: 720 / 32 = 22.5 (not a whole number).
TARGET_W, TARGET_H = 704, 1280

in_pattern  = str(LOCAL_RES_DIR / "frame_%05d.png")
out_pattern = str(LOCAL_DS_DIR / "frame_%05d.png")

# Safety check: ensure pattern matches at least one file
if not any(LOCAL_RES_DIR.glob("frame_*.png")):
  raise FileNotFoundError(f"No files like frame_*.png found in {LOCAL_RES_DIR}")

# Batch-scale with lanczos (excellent downscale)
!ffmpeg -y -hide_banner -loglevel error \
  -i "{in_pattern}" \
  -vf "scale={TARGET_W}:{TARGET_H}:flags=lanczos" \
  "{out_pattern}"

print(f"Downscaled frames written to: {LOCAL_DS_DIR}")

## Step 5.2b — Compress and save frames to Drive

The restored (and optionally downscaled) frames are archived into a `.tar.gz` file and moved to the Rekrea environment in Drive. This serves two purposes:

1. **Handoff to interpolation:** the RIFE notebook reads the archive from Drive.
2. **Persistence:** if the Colab session expires, the work is not lost.

The frame rate (`cfr`) is **embedded in the archive filename** (e.g. `restored_frames_fps_30.tar.gz`). This allows the correct frame rate to be recovered from the filename when the archive is loaded in a new session — without needing to re-probe the original video.

In [ ]:
# 📁 Step 5.2b: Compress and move output to Google Drive
from pathlib import Path

LOCAL_DS_DIR = Path("/content/restored_frames_ds")
DRV_RES_DIR = Path("/content/drive/MyDrive/Rekrea/VideoRestoration/output")
DRV_DS_DIR = Path("/content/drive/MyDrive/Rekrea/VideoRestoration/downscaled")

# Create target directory
DRV_RES_DIR.mkdir(parents=True, exist_ok=True)

# Embedding frame rate in tar name
tar_name = f"restored_frames_fps_{int(cfr)}.tar.gz"

# Compress the frames and move to Drive
!tar -czf "{tar_name}" -C /content/restored_frames .
!mv "{tar_name}" "{DRV_RES_DIR}/"

print(f"Saved: {DRV_RES_DIR / tar_name}")

# Compress and move downscaled frames if exist
if LOCAL_DS_DIR.is_dir():
  # Create target directory
  DRV_DS_DIR.mkdir(parents=True, exist_ok=True)

  # Embedding frame rate in tar name
  tar_name_ds = f"restored_frames_fps_{int(cfr)}_ds.tar.gz"

  # Compress the frames and move to Drive
  !tar -czf "{tar_name_ds}" -C /content/restored_frames_ds .
  !mv "{tar_name_ds}" "{DRV_DS_DIR}/"

  print(f"Saved: {DRV_DS_DIR / tar_name_ds}")

## Step 6 — Frame interpolation [Separate notebook]

Frame interpolation is implemented in the **RIFE interpolation notebook** and is not run here. The cell below is a placeholder showing what the integration looks like.

**What RIFE does:** RIFE (Real-Time Intermediate Flow Estimation) synthesises new frames between existing ones using optical flow. Doubling the frame count (2× interpolation) makes motion appear smoother — turning 30 fps footage into a 60 fps feel. The "Real-Time" in the name refers to the model's ability to run at interactive speeds on a GPU.

**To perform interpolation:**
1. Ensure Step 5.2b has saved the frames archive to Drive.
2. Open the RIFE interpolation notebook.
3. It reads the `restored_frames_fps_*.tar.gz` archive from Drive, runs interpolation, and saves `interpolated_frames_fps_*.tar.gz` back to Drive.
4. Return to this notebook and continue at Step 7.

**If you are not performing interpolation:** skip directly to Step 8 (Reassemble Final Video). The `cfr` variable is already set from Step 2.2.

In [ ]:
# ⏩ Step 6: Frame interpolation (Optional)
#%cd ..
#!git clone https://github.com/hzwer/arXiv2020-RIFE.git
#%cd arXiv2020-RIFE
#!pip install -r requirements.txt
#!python inference_video.py --img ../enhanced_frames --output ../interpolated_frames --exp 2

## Step 7.1 — Load processed frames from Drive (optional)

This step is only needed when **returning to the notebook in a new session** after interpolation, or when the local frames were lost after a runtime reset.

The cell automatically selects the most recent matching archive using a priority order:
- If interpolated frames exist in Drive → load those (preferred, with their embedded FPS).
- Otherwise → fall back to the restored frames archive.

The FPS value is recovered from the filename pattern `*_fps_30.tar.gz`, making each archive self-describing without needing to re-probe the original video.

> If local frames already exist from the current session (Step 4 or Step 7.2), skip this cell to avoid overwriting them.

In [ ]:
# 📁 Step 7.1 (Optional) Upload frames for video reassembling 
from google.colab import drive
from pathlib import Path
import re

# Mount only if needed
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
   drive.mount("/content/drive")

DRV_RES_DIR = Path("/content/drive/MyDrive/Rekrea/VideoRestoration/output")
DRV_INT_DIR = Path("/content/drive/MyDrive/Rekrea/VideoInterpolation/output")
LOCAL_OUT_DIR  = Path("/content/enhanced_frames")
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Upload interpolated or restored frames
if DRV_INT_DIR.is_dir():
  pattern = "interpolated_frames_fps_*.tar.gz"
  candidates = sorted(
      DRV_INT_DIR.glob(pattern),
      key=lambda p: p.stat().st_mtime,
      reverse=True
  )
  
  TAR_PATH = candidates[0]  # newest match
  print("Using:", TAR_PATH)

  # Extract FPS back from filename
  match = re.search(r"fps_(\d+)\.tar\.gz$", TAR_PATH.name)

  if match:
      cfr = int(match.group(1))
      print("Recovered FPS from filename:", cfr)

  # -x extract, -z gzip, -f file, -C output directory
  !tar -xzf "{TAR_PATH}" -C "{LOCAL_OUT_DIR}"

  print("Done. Extracted into:", LOCAL_OUT_DIR)

  if not candidates:
      raise FileNotFoundError("No matching interpolated_frames archives found.")

elif DRV_RES_DIR.is_dir():
  pattern = "restored_frames_fps_*.tar.gz"
  candidates = sorted(
      DRV_RES_DIR.glob(pattern),
      key=lambda p: p.stat().st_mtime,
      reverse=True
  )

  TAR_PATH = candidates[0]  # newest match
  print("Using:", TAR_PATH)

  # Extract FPS back from filename
  match = re.search(r"fps_(\d+)\.tar\.gz$", TAR_PATH.name)
  
  if match:
      cfr = int(match.group(1))
      print("Recovered FPS from filename:", cfr)

  # -x extract, -z gzip, -f file, -C output directory
  !tar -xzf "{TAR_PATH}" -C "{LOCAL_OUT_DIR}"

  print("Done. Extracted into:", LOCAL_OUT_DIR)

  if not candidates:
     raise FileNotFoundError("No matching interpolated_frames or restored_frames archives found.")

## Step 7.2 — Superscaling with Real-ESRGAN (optional)

Superscaling applies Real-ESRGAN a **second time** — after interpolation — to recover resolution on frames that were downscaled in Step 5.2a, or to push the final output to a higher resolution.

**When to use it:**
- You downscaled frames before interpolation (Step 5.2a) and want to recover the lost resolution on the final output.
- You want a higher output resolution than the single-pass restoration produced.

**Important constraints:**
- Use `--outscale 2` unless you have more than 16 GB VRAM — larger scales cause out-of-memory errors on most Colab GPUs.
- Real-ESRGAN supports `--tile 1024` or `--tile 512` to process the image in patches if full-frame inference runs out of memory.
- Applying superscaling to already-upscaled frames (without prior downscale) rarely improves quality and wastes time.

### **Important Note:** If the runtime was restarted, run Step 3 to setup Real-ESGRAN

In [ ]:
# 📁 Step 7.2 (Optional) Upscale using Real-ESGRAN
LOCAL_OUT_DIR  = Path("/content/upscaled_frames")
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 🚀 Superscaling with Real-ESGRAN
!python inference_realesrgan.py -n RealESRGAN_x4plus -i /content/enhanced_frames/ -o /content/upscaled_frames/ --outscale 3 --tile 1024

# Release GPU Memory
import torch
torch.cuda.empty_cache()

## Step 8.1a — Reassemble the final video

FFmpeg re-encodes the processed frames into an MP4. Key parameters:

| Parameter | Value | Meaning |
|---|---|---|
| `vcodec=libx264` | H.264 | Most compatible codec — works in browsers, phones, and all players |
| `crf=16` | quality | Constant Rate Factor: 0 = lossless, 51 = worst. 16 gives near-lossless quality at a manageable file size |
| `preset=slow` | encoding effort | Slower presets achieve better compression at the same CRF — no quality loss, smaller file |
| `pix_fmt=yuv420p` | pixel format | Required for H.264 compatibility |
| `movflags=+faststart` | streaming | Moves the index to the start of the file so the video can start playing while downloading |

The `cfr` variable must be set — the cell raises an explicit error if it is `None` (VFR suspected) to prevent producing a video with incorrect timing.

In [ ]:
# 🎬 Step 8.1a: Reassemble Final Video
import ffmpeg
from pathlib import Path

# Safety: CFR must be known
if cfr is None:
    raise ValueError("cfr is None (VFR suspected). Normalize to CFR first or set a target FPS explicitly.")

OUTPUT_VIDEO = "output_video.mp4"

# The output pattern example: restored_frames/frame_00001.png
input_pattern = "restored_frames/frame_%05d.png"

# Sanity check that at least one frame exists
if not any(Path("restored_frames").glob("frame_*.png")):
    raise FileNotFoundError("No restored output frames found matching restored_frames/frame_*.png")

(
    ffmpeg
    .input(input_pattern, framerate=cfr)  # if your numbering starts at 0, add start_number=0
    .output(
        OUTPUT_VIDEO,
        vcodec="libx264",
        crf=16,
        preset="slow",
        pix_fmt="yuv420p",
        movflags="+faststart",
    )
    .run(overwrite_output=True)
)

## Step 8.1b — Video stabilisation (optional)

`vidstab` (FFmpeg's `vidstabdetect` / `vidstabtransform` filter pair) reduces camera shake in two passes:

1. **Pass 1** (`vidstabdetect`): analyses motion between frames and writes a motion data file (`transforms.trf`). `shakiness=10` sets how aggressively to detect motion (1–10); `accuracy=15` sets detection precision.
2. **Pass 2** (`vidstabtransform`): applies compensating transforms to smooth the camera path. `smoothing=30` controls how many frames ahead/behind are used when planning the smoothed path — higher values give more fluid but more cropped results.

> Stabilisation is most useful when upscaling has amplified existing camera shake. It is not a substitute for stable capture technique — extreme shake will result in significant border cropping.

In [ ]:
# ✅ Optional: Video Stabilization
!ffmpeg -i output_video.mp4 -vf vidstabdetect=shakiness=10:accuracy=15 -f null -
!ffmpeg -i output_video.mp4 -vf vidstabtransform=smoothing=30:input=transforms.trf -c:v libx264 -preset slow -crf 18 -c:a copy stabilized_video.mp4

## Step 8.2 — Reattach audio from the original video

The entire processing pipeline works on **image frames** — the audio track is never touched and is effectively discarded when we extract PNGs in Step 2.1. This step restores it.

### Why audio is lost

FFmpeg's frame extraction outputs individual image files with no concept of audio. The restored and reassembled video in Step 8.1 is a **silent video**. The audio track must be taken from the original source and combined with the new video stream.

### What muxing means

**Muxing** (multiplexing) is the process of combining separate streams — video, audio, subtitles — into a single container file. It is the opposite of *demuxing* (splitting a container into its separate streams).

The `mux_video_and_audio` function is equivalent to this FFmpeg command:
```
  ffmpeg -i output_video.mp4 -i input_video.mp4 \
    -map 0:v:0 -map 1:a:0 \
    -c:v copy -shortest \
    output_video_waudio.mp4
```

| Element | Meaning |
|---|---|
| `-map 0:v:0` | Take the **video** stream from input 0 (the restored video) |
| `-map 1:a:0` | Take the **audio** stream from input 1 (the original video) |
| `-c:v copy` | Copy the video stream without re-encoding — no quality loss, near-instant |
| `-shortest` | Stop at the shorter stream's end — handles small duration drift between restored and original |

> **Note:** If the restored video is slightly longer or shorter than the original (due to frame count changes from interpolation), `-shortest` prevents a desync tail. For large duration differences (e.g. after 2× interpolation that doubled the frame count), the audio will need to be stretched or the video trimmed — this is not handled automatically here.

### Important Note: If video stabilization was performed, modify the value of the target video as:
```
  INPUT_VIDEO = stabilized_video.mp4
```

In [ ]:
# 🔊 Step 8.2: Reattach audio from original video
from google.colab import files
from pathlib import Path

INPUT_AUDIO = "input_video.mp4"
INPUT_VIDEO = "output_video.mp4"
OUTPUT_VIDEO = "output_video_waudio.mp4"
RES_DIR = Path("/content/drive/MyDrive/Rekrea/VideoRestoration")
DRV_AUDIO_PATH = Path("/content/drive/MyDrive/Rekrea")  / INPUT_AUDIO
DRV_VIDEO_PATH = RES_DIR  / INPUT_VIDEO
DRV_OUTPUT_PATH = RES_DIR  / OUTPUT_VIDEO
LOCAL_AUDIO_PATH = Path("/content") / INPUT_AUDIO
LOCAL_VIDEO_PATH = Path("/content") / INPUT_VIDEO

# Function to attach audio from source video to target video 
def mux_video_and_audio(
    input_video_file: str,
    input_audio_file: str,
    output_file: str,
    overwrite: bool = True,
):
    """
    Equivalent to:
      ffmpeg -i INPUTVIDEO -i INPUTAUDIO -c:v copy -map 0:v:0 -map 1:a:0 -shortest OUTPUT
    """
    in_video = ffmpeg.input(input_video_file)   # input 0
    in_audio = ffmpeg.input(input_audio_file)   # input 1

    v = in_video.video                          # -map 0:v:0
    a = in_audio.audio                          # -map 1:a:0

    (
        ffmpeg
        .output(
            v, a,
            output_file,
            vcodec="copy",      # -c:v copy
            acodec="copy",      # keep audio as-is (optional but typical)
            shortest=None       # -shortest (flag)
        )
        .run(overwrite_output=overwrite)
    )

# Prepare audio source and target video 
if LOCAL_AUDIO_PATH.exists() and LOCAL_VIDEO_PATH.exists():
    print("Using existing audio input:", LOCAL_AUDIO_PATH, "and video input", LOCAL_VIDEO_PATH)
else:
    print("No audio input and/or video input found. Uploading from Rekrea environment.")
    !cp "{DRV_AUDIO_PATH}" "{LOCAL_AUDIO_PATH}"
    !cp "{DRV_VIDEO_PATH}" "{LOCAL_VIDEO_PATH}"

print("Local working copies ready.")

# Reattach audio to output video
mux_video_and_audio(INPUT_VIDEO, INPUT_AUDIO, OUTPUT_VIDEO)

print("Done. Audio attached to:", OUTPUT_VIDEO)

# Move output file to the Rekrea environment in Drive
!cp "{OUTPUT_VIDEO}" "{DRV_OUTPUT_PATH}"

print("Persistent file stored in Rekrea environment in Drive.")

## Reflection — What could be improved?

| Limitation | Possible solution |
|---|---|
| VFR video raises an error at reassembly | Add CFR normalisation using FFmpeg's `fps` filter before frame extraction |
| Two-pass workflow requires manual steps between notebooks | A shared Drive config file (JSON) could carry `cfr`, paths, and processing state between notebooks automatically |
| ESRGAN output can look over-sharpened or have GAN artefacts | Try `RealESRNet_x4plus` for a more conservative output, or reduce `--outscale` |
| `--face_enhance` adds time even for non-portrait content | Remove the flag — GFPGAN only adds value for facial close-ups |
| Large PNG archives on Drive | Use FFV1 lossless video instead of individual PNGs to reduce archive size by 2–4× |
| `--outscale 4` runs out of VRAM | Use `--tile 512` — Real-ESRGAN processes the image in overlapping patches to stay within VRAM |
| Stabilisation crops the frame borders | Reduce `smoothing` value, or add padding before stabilisation and crop after |

Real-ESRGAN is the foundation of several commercial upscaling tools (Topaz Video AI, Gigapixel AI). The key differences between this open-source implementation and commercial products are the post-processing pipeline, the volume and diversity of training data, and hardware-specific optimisations.